<a href="https://colab.research.google.com/github/asad1gove/bio-inspired-systems-lab/blob/main/greyWolf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

NUM_TEST_CASES = 50
BUDGET = 120

test_costs = np.random.randint(5, 25, size=NUM_TEST_CASES)

NUM_REQUIREMENTS = 30
test_coverage_matrix = np.random.randint(0, 2, size=(NUM_TEST_CASES, NUM_REQUIREMENTS))

print(f"Total available test cases: {NUM_TEST_CASES}")
print(f"Budget limit: {BUDGET} units of time")
print(f"Total unique requirements to cover: {NUM_REQUIREMENTS}")

Total available test cases: 50
Budget limit: 120 units of time
Total unique requirements to cover: 30


In [8]:
def fitness_function(position):
    selection = (1 / (1 + np.exp(-position))) > 0.5

    total_cost = np.sum(test_costs[selection])

    if np.sum(selection) == 0:
        return 0.0

    covered_reqs = np.any(test_coverage_matrix[selection], axis=0)
    coverage_ratio = np.sum(covered_reqs) / NUM_REQUIREMENTS

    fitness = coverage_ratio * 1000

    if total_cost > BUDGET:
        penalty = (total_cost - BUDGET) * 50
        fitness -= penalty
    else:
        fitness += (BUDGET - total_cost) * 0.5

    return fitness

In [9]:
def grey_wolf_optimizer(num_wolves=20, max_iter=100):
    Positions = np.random.uniform(-4, 0, (num_wolves, NUM_TEST_CASES))

    Alpha_pos = np.zeros(NUM_TEST_CASES)
    Alpha_score = -float('inf')

    Beta_pos = np.zeros(NUM_TEST_CASES)
    Beta_score = -float('inf')

    Delta_pos = np.zeros(NUM_TEST_CASES)
    Delta_score = -float('inf')

    convergence_curve = []

    for l in range(max_iter):
        for i in range(num_wolves):
            Positions[i] = np.clip(Positions[i], -10, 10)

            fitness = fitness_function(Positions[i])

            if fitness > Alpha_score:
                Delta_score = Beta_score
                Delta_pos = Delta_pos.copy()
                Beta_score = Alpha_score
                Beta_pos = Alpha_pos.copy()
                Alpha_score = fitness
                Alpha_pos = Positions[i].copy()

            elif fitness > Beta_score:
                Delta_score = Beta_score
                Delta_pos = Beta_pos.copy()
                Beta_score = fitness
                Beta_pos = Positions[i].copy()

            elif fitness > Delta_score:
                Delta_score = fitness
                Delta_pos = Positions[i].copy()

        a = 2 - l * (2 / max_iter)

        for i in range(num_wolves):
            for j in range(NUM_TEST_CASES):
                r1, r2 = np.random.rand(), np.random.rand()
                A1 = 2 * a * r1 - a
                C1 = 2 * r2
                D_alpha = abs(C1 * Alpha_pos[j] - Positions[i, j])
                X1 = Alpha_pos[j] - A1 * D_alpha

                r1, r2 = np.random.rand(), np.random.rand()
                A2 = 2 * a * r1 - a
                C2 = 2 * r2
                D_beta = abs(C2 * Beta_pos[j] - Positions[i, j])
                X2 = Beta_pos[j] - A2 * D_beta

                r1, r2 = np.random.rand(), np.random.rand()
                A3 = 2 * a * r1 - a
                C3 = 2 * r2
                D_delta = abs(C3 * Delta_pos[j] - Positions[i, j])
                X3 = Delta_pos[j] - A3 * D_delta

                Positions[i, j] = (X1 + X2 + X3) / 3

        convergence_curve.append(Alpha_score)

    return Alpha_pos, Alpha_score, convergence_curve

In [15]:
best_position, best_score, curve = grey_wolf_optimizer(num_wolves=30, max_iter=80)

final_selection = (1 / (1 + np.exp(-best_position))) > 0.5
selected_indices = np.where(final_selection)[0]
total_time_used = np.sum(test_costs[final_selection])
covered_reqs = np.any(test_coverage_matrix[final_selection], axis=0)

print(f"Selected Test Cases : {selected_indices.tolist()}")
print(f"Total Cost          : {total_time_used} / {BUDGET} units")
print(f"Requirements Covered: {np.sum(covered_reqs)} / {NUM_REQUIREMENTS}")
print(f"Best Fitness Score  : {best_score:.3f}")

Selected Test Cases : [4, 20, 45]
Total Cost          : 33 / 120 units
Requirements Covered: 30 / 30
Best Fitness Score  : 1043.500
